In [5]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    api_key="ai",
    model="openai/gpt-oss-20b",
    base_url="http://192.168.0.110:8000/v1",
)

In [2]:
from typing import List, Annotated
from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage

class AgentState(TypedDict):
    messages: Annotated[List[AnyMessage], lambda x, y: x + y]
    turn_count: int

In [3]:
def chatbot_node(state: AgentState):
    print("--챗봇 노드 실행--")
    messages = state['messages']
    turn_count = state['turn_count']

    response = llm.invoke(messages)

    return {
        "messages": [response],
        "turn_count": turn_count
    }

In [14]:
def chatbot_node2(state: AgentState):
    last_answer = state['messages'][-1]
    new_prompt = {
        "messages": [HumanMessage(content="나는 평소에 데드리프트를 즐겨하는데부상을 방지할 수 있는 팁이 있나요?")],
    }
    res = llm.invoke(new_prompt)
    return {
        "messages": [response],
        "turn_count": turn_count
    }

In [15]:
from langgraph.graph import StateGraph, END

graph_builder = StateGraph(AgentState)

graph_builder.add_node("chatbot", chatbot_node)
graph_builder.add_node("chatbot2", chatbot_node2)

graph_builder.set_entry_point("chatbot")
graph_builder.add_edge("chatbot", "chatbot2")
graph_builder.add_edge("chatbot", END)

agent_app = graph_builder.compile()

In [16]:
from langchain_core.messages import HumanMessage, AIMessage

initial_input = {
    "messages": [HumanMessage(content="안녕하세요 이창수라고 합니다.")],
    "turn_count": 0
}

next_input = {
    "messages": [HumanMessage(content="나는 평소에 데드리프트를 즐겨하는데부상을 방지할 수 있는 팁이 있나요?")],
    "turn_count": 0
}

final_state = agent_app.invoke(initial_input)

print("\n--최종대화기록--")
for message in final_state['messages']:
    if isinstance(message, HumanMessage):
        print(f"사용자: {message.content}")
    elif isinstance(message, AIMessage):
        print(f"AI 응답: {message.content}")


--챗봇 노드 실행--


ValueError: Invalid input type <class 'dict'>. Must be a PromptValue, str, or list of BaseMessages.